# Participant starter: build your submission (Track A)

This notebook loads the provided **simulated** scenes and their **real** counterparts,
shows you what each looks like, and then writes a `submissions/<your_team_name>/` folder
in the layout the evaluator expects.

Your job is to fill in the **translation technique** cell below: given a simulated image
(and its scene label), produce a generated image that should look real while keeping the same objects and geometry as the simulated source.

See `README_CONTESTANTS.md` for the exact submission rules (file naming, allowed formats,
what you may and may not change). Read that file before you submit.

In [ ]:
import sys
from pathlib import Path

VENV_PYTHON = Path(sys.executable).resolve()
if Path(sys.prefix).resolve() == Path(sys.base_prefix).resolve():
    raise RuntimeError(
        'Select the Track A virtual-environment kernel (.venv or venv) and restart the notebook.'
    )

print('Using environment:', VENV_PYTHON)

RuntimeError: Select the Sim2Real (.venv) kernel and restart the notebook.

In [ ]:
import json
import shutil

from huggingface_hub import snapshot_download
from PIL import Image

# Change this to your team name. This becomes the folder name under submissions/.
TEAM_NAME = 'your_team_name'
DATASET_ID = 'ISU-Test/isu-challenge-dataset'

TRACK_A_DIR = Path.cwd() if Path.cwd().name == 'track_a' else Path.cwd() / 'track_a'
REPO_ROOT = TRACK_A_DIR.parent
DATA_DIR = REPO_ROOT / 'data'
SUBMISSION_DIR = TRACK_A_DIR / 'submissions' / TEAM_NAME

# Download the dataset outside track_a when it is not already available locally.
if not any((DATA_DIR / 'images').glob('*_sim.png')):
    print(f'Downloading {DATASET_ID} to {DATA_DIR}...')
    snapshot_download(
        repo_id=DATASET_ID,
        repo_type='dataset',
        local_dir=str(DATA_DIR),
    )

print(f'Using data directory: {DATA_DIR}')


NameError: name 'Path' is not defined

## 1. Load the provided data

The notebook loads data from the repository-level `data/` folder, which is outside `track_a/`. If the folder does not contain simulated scenes, the setup cell downloads `ISU-Test/isu-challenge-dataset` from Hugging Face automatically.

The data contains simulated source images, real target-domain reference images, and scene labels, organized in subfolders:

- `data/real/<stem>.jpg` — a **real** target-domain reference photo
- `data/images/<stem>_sim.png` — a **simulated** source render you must translate (one output is required for each simulated image)
- `data/labels/<stem>_label.json` — categorical scene labels used by the evaluator

Real reference images are used as a target distribution. They do not need to match simulated images one-to-one or share their filename stems.

The real and simulated images are not pixel-aligned crops of each other and may have different resolutions. They depict comparable in-car scenes rendered by the simulator and captured with a real camera.


In [ ]:
def discover_scenes(data_dir):
    scenes = {}
    image_dir = data_dir / 'images'
    label_dir = data_dir / 'labels'

    for sim_path in sorted(image_dir.glob('*_sim.png')):
        stem = sim_path.stem[:-len('_sim')]
        label_path = label_dir / f'{stem}_label.json'
        if label_path.exists():
            scenes[stem] = {
                'simulated': sim_path,
                'label': label_path,
            }
    return scenes


def discover_reference_images(data_dir):
    real_dir = data_dir / 'real'
    return sorted(real_dir.glob('*.jpg'))


scenes = discover_scenes(DATA_DIR)
reference_images = discover_reference_images(DATA_DIR)

print(f'Found {len(scenes)} simulated scenes: {list(scenes)}')
print(f'Found {len(reference_images)} real reference images.')


NameError: name 'DATA_DIR' is not defined

In [ ]:
# Sanity check: look at one scene before writing any code.
example_stem = next(iter(scenes))
example = scenes[example_stem]
sim_image = Image.open(example['simulated']).convert('RGB')
label = json.loads(example['label'].read_text())

if reference_images:
    real_image = Image.open(reference_images[0]).convert('RGB')

    # Side-by-side preview, simulated then one real reference, scaled to the same height.
    height = 360
    sim_preview = sim_image.resize((int(sim_image.width * height / sim_image.height), height))
    real_preview = real_image.resize((int(real_image.width * height / real_image.height), height))
    preview = Image.new('RGB', (sim_preview.width + real_preview.width + 10, height), 'white')
    preview.paste(sim_preview, (0, 0))
    preview.paste(real_preview, (sim_preview.width + 10, 0))
    print(f'{example_stem}: simulated {sim_image.size} (left) vs real reference {real_image.size} (right)')
    display(preview)
else:
    print(f'{example_stem}: simulated {sim_image.size}; no real reference images found')

display(label)

## 2. Your translation technique

Implement `translate` below. It receives:

- one simulated image 
- its label dict (in case you
want to condition on it)
- **kwargs (e.g., channel information such as, seg maps, canny edges, or depht maps) 

and must return one PIL image that:

- keeps the same width, height, framing, and camera viewpoint as the simulated input
  (resizing is allowed but reduces fidelity/semantic scores — see `README_CONTESTANTS.md`)
- does not crop, rotate, mirror, or change scene geometry
- does not add, remove, or move scene objects
- is RGB

The placeholder below just returns the simulated image unchanged, so the rest of this notebook
runs end to end out of the box and produces a (trivial, bad) baseline submission. Replace the
body with your own model / pipeline.

In [ ]:
def translate(simulated_image: Image.Image, stem: str, label: dict, **kwargs) -> Image.Image:
    """Translate one simulated scene into a real-looking one."""
    # >>> REPLACE THIS WITH YOUR TECHNIQUE <<<
    # This placeholder is the identity function: it returns the simulated image
    # unchanged, which will score poorly on realism but perfectly on semantic
    # preservation. Use it to confirm the pipeline runs before you plug in a model.
    return simulated_image.convert('RGB')

## 3. Write the submission folder

This builds `submissions/<team_name>/` with the four subfolders the evaluator expects:
`simulated/` and `reference/` are copies of the provided data, `labels/` is a copy of the scene
labels, and `generated/` holds your `translate()` output for every scene. Do not hand-edit
`simulated/` or `reference/` after this runs.

In [ ]:
for name in ('simulated', 'generated', 'reference', 'labels'):
    (SUBMISSION_DIR / name).mkdir(parents=True, exist_ok=True)

# Copy the real images as a separate reference distribution; they do not pair one-to-one with scenes.
for reference_path in reference_images:
    shutil.copy(reference_path, SUBMISSION_DIR / 'reference' / reference_path.name)

for index, (stem, paths) in enumerate(scenes.items(), 1):
    print(f'{index}/{len(scenes)}: {stem}')
    label = json.loads(paths['label'].read_text())
    simulated_image = Image.open(paths['simulated']).convert('RGB')

    generated_image = translate(simulated_image, stem, label)
    generated_image.save(SUBMISSION_DIR / 'generated' / f'{stem}.png')

    shutil.copy(paths['simulated'], SUBMISSION_DIR / 'simulated' / paths['simulated'].name)
    shutil.copy(paths['label'], SUBMISSION_DIR / 'labels' / paths['label'].name)

print(f'\nWrote submission to {SUBMISSION_DIR}')

## 4. Next steps

Run `track_a_challenge_evaluator.ipynb` to score `submissions/<your_team_name>/` alongside any
other teams. Re-run this notebook (with your real `translate()` implementation) any time you
want to regenerate your submission before scoring.